# 07. RL reward backend

The optional reinforcement-learning backend turns a candidate rule into a
scalar reward. `RuleSimulator` is an `Evaluator` (component 03's protocol), so
it drops straight into `run_self_play` as the `simulator`. Scoring one rule is
a four-step pipeline:

1. `env = gym.make(env_name)` builds the base environment.
2. `transformer(env, rule_string)` wraps it so each observation says whether the
   rule currently matches the recent event history (`HistoryToRuleWrapperBase`).
3. `agent_builder(wrapped)` trains an agent in that wrapped env
   (`q_learning_agent_builder`).
4. `agent_eval(agent, wrapped)` plays the trained agent and returns the score.

This needs the `[rl]` extra (`gymnasium`). The real reward comes from an
OpenTheChests environment, installed separately, so the search core never
imports it. This notebook uses a small **self-contained environment** in the
same spirit, trains a real Q-agent (no mocked rewards), and ends by letting MCTS
and then `train()` discover the rules from the reward alone.

In [1]:
import alpha_rule
import gymnasium as gym
print("alpha_rule", alpha_rule.__version__, "| gymnasium", gym.__version__)

alpha_rule 0.1.0 | gymnasium 1.2.3


## Elements

`alpha_rule.evaluation`
* `RuleSimulator(env_name, agent_builder, transformer, agent_eval, *, reward_scale=None, agent_builder_kwargs=None, seed=None)`:
  an `Evaluator`. `evaluate(node)` strips a trailing `<END>`, runs the four-step
  pipeline, and returns the agent's score. The base env is built once (lazy
  `gym.make`) and reused. `reward_scale` is the positive cap the training stack
  reads to scale value targets; `agent_builder_kwargs` are forwarded to the
  builder each call (the per-leaf training budget); `seed` re-seeds the base env
  per evaluate for reproducible scores.

`alpha_rule.wrappers.history_to_rule`
* `HistoryToRuleWrapperBase(env, rule_list, window=15, *, strip_end_marker=False)`:
  an `ObservationWrapper` whose observation is a binary vector, one entry per
  rule, that is 1 when the rule matches the rolling window. Matching is anchored
  at the newest event, so a rule fires only when its last event is the most
  recent one.

`alpha_rule.policy_agents.q_learning`
* `q_learning_agent_builder(env, total_timesteps=1000, *, seed=None, epsilon=0.1, ...)`:
  trains a tabular Q-agent, returns `(q_table, policy)`.
* `q_learning_agent_eval_mean_reward_success_steps(agent, env, n_eval_episodes=50)`:
  returns `(mean_reward, success_rate, mean_steps)`, or scalar `-inf` when the
  candidate rule never fires (a structural failure the search prunes).
* `get_state_tuple`, `q_learning_agent_eval_total_reward`,
  `..._recognition_distance`, `..._ground_truth_distance`: more state/eval helpers.

## A realistic environment

`BurstChestEnv` emits a repeating burst of **3 A's then 4 B's** (period 7). Two
chests sit behind that stream:

* the **A-chest** becomes ready on the 3rd A of each burst, then resets;
* the **B-chest** becomes ready on the 4th B of each burst, then resets.

Actions are `0` (wait), `1` (open the A-chest), `2` (open the B-chest). Opening a
chest on its ready step earns `+1`; a mistimed open costs `-0.25`; waiting is
`0`. The observable `get_observations()["open"]` does **not** reveal readiness,
so the agent's only timing signal is the candidate rule's match bit. A rule that
captures a chest's burst lets the agent time its opens; a rule that does not
cannot. That is what makes a rule's score meaningful.

In [2]:
import numpy as np
from gymnasium import spaces

PATTERN = [0, 0, 0, 1, 1, 1, 1]      # A A A B B B B  (e_type 0 = A, 1 = B)
A_READY_POS, B_READY_POS = 2, 6      # ready on the last event of each burst


class _Box:
    def __init__(self):
        self._ready = False
    def is_ready(self):
        return self._ready


class _OTC:
    """OpenTheChests-style backend. ``get_observations()['open']`` is the
    observable box state; it stays zero here so the agent must time its opens
    from the rule, not from a readiness oracle. ``boxes[i].is_ready()`` is the
    ground-truth signal the env uses to pay reward."""
    def __init__(self):
        self.boxes = [_Box(), _Box()]
    def get_observations(self):
        return {"open": (0, 0)}


class BurstChestEnv(gym.Env):
    """Emits 3 A's then 4 B's, repeating; chests become ready at the end of
    each burst. Self-contained stand-in for an OpenTheChests env."""

    def __init__(self, episode_len=28):
        super().__init__()
        self.action_space = spaces.Discrete(3)         # 0 wait, 1 open A, 2 open B
        self.observation_space = spaces.Dict({
            "e_type": spaces.Discrete(2),
            "start": spaces.Box(0.0, 1e9, shape=(1,)),
            "end": spaces.Box(0.0, 1e9, shape=(1,)),
        })
        self._otc = _OTC()
        self._pos = 0
        self._t = 0
        self._clock = 0.0
        self._episode_len = episode_len

    def get_types(self):
        return ["A", "B"]

    def get_otc(self):
        return self._otc

    def _emit(self):
        pos = self._pos % len(PATTERN)
        etype = PATTERN[pos]
        self._otc.boxes[0]._ready = (pos == A_READY_POS)
        self._otc.boxes[1]._ready = (pos == B_READY_POS)
        start = self._clock
        self._clock += 2.0
        self._pos += 1
        return {"e_type": etype, "start": np.array([start]), "end": np.array([start + 1.0])}

    def _reward(self, action):
        if action == 1:
            return 1.0 if self._otc.boxes[0].is_ready() else -0.25
        if action == 2:
            return 1.0 if self._otc.boxes[1].is_ready() else -0.25
        return 0.0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._pos = 0
        self._t = 0
        self._clock = 0.0
        return self._emit(), {}

    def step(self, action):
        # Reward against the readiness the agent just observed, THEN advance.
        reward = self._reward(action)
        self._t += 1
        truncated = self._t >= self._episode_len
        obs = self._emit()
        return obs, reward, False, truncated, {}


print("BurstChestEnv ready; types:", BurstChestEnv().get_types(),
      "| actions: 0=wait 1=openA 2=openB")

BurstChestEnv ready; types: ['A', 'B'] | actions: 0=wait 1=openA 2=openB


## The event stream and chest readiness

Rolling the raw env shows the burst and exactly when each chest is ready (the
last event of each burst).

In [3]:
env = BurstChestEnv()
obs, _ = env.reset()
print("pos event  A_ready B_ready")
for i in range(14):
    a_ready = env.get_otc().boxes[0].is_ready()
    b_ready = env.get_otc().boxes[1].is_ready()
    etype = env.get_types()[int(obs["e_type"])]
    print(f"{i:3d}  {etype}        {int(a_ready)}       {int(b_ready)}")
    obs, _, _, _, _ = env.step(0)

pos event  A_ready B_ready
  0  A        0       0
  1  A        0       0
  2  A        1       0
  3  B        0       0
  4  B        0       0
  5  B        0       0
  6  B        0       1
  7  A        0       0
  8  A        0       0
  9  A        1       0
 10  B        0       0
 11  B        0       0
 12  B        0       0
 13  B        0       1


## The candidate rule as a detector

A rule for "3 A's in a row" (`A A < A <`, built by the grammar) fires **exactly**
on the step the A-chest becomes ready, because matching is anchored at the newest
event. Wrapping the env with that rule, the match bit lines up with the readiness
column above.

In [4]:
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.wrappers.history_to_rule import HistoryToRuleWrapperBase

g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<",))

def build_rule(seq):
    node = g.root()
    for name in seq:
        prods = {p.name: p for p in g.applicable_productions(node)}
        node = g.apply(node, prods[name])
    return node.name

def build_n(event_type, n, max_steps=40):
    """Greedily build an n-event single-type rule (add the event when it is
    applicable, else add '<', stop once n events are placed and the rule can
    terminate)."""
    node = g.root()
    placed = 0
    for _ in range(max_steps):
        ap = {p.name: p for p in g.applicable_productions(node)}
        if placed < n and event_type in ap:
            node = g.apply(node, ap[event_type]); placed += 1
        elif placed >= n and "END_RULE" in ap:
            break
        elif "<" in ap:
            node = g.apply(node, ap["<"])
        else:
            break
    return node.name

RULE_A = build_rule(["A", "A", "<", "A", "<"])      # 3 A's
print("candidate rule (3 A's):", repr(RULE_A))

WINDOW = 4
base = BurstChestEnv()
wrapped = HistoryToRuleWrapperBase(base, rule_list=[RULE_A], window=WINDOW)
match, _ = wrapped.reset()      # wrapper returns the gym (obs, info) tuple
print("\npos event  rule_match  A_ready")
for i in range(14):
    a_ready = int(base.get_otc().boxes[0].is_ready())
    etype = "A" if PATTERN[i % len(PATTERN)] == 0 else "B"
    flag = " <= fires" if match[0] == 1 else ""
    print(f"{i:3d}  {etype}        {int(match[0])}           {a_ready}{flag}")
    match, _, _, _, _ = wrapped.step(0)

candidate rule (3 A's): 'A A < A <'

pos event  rule_match  A_ready
  0  A        0           0
  1  A        0           0
  2  A        1           1 <= fires
  3  B        0           0
  4  B        0           0
  5  B        0           0
  6  B        0           0
  7  A        0           0
  8  A        0           0
  9  A        1           1 <= fires
 10  B        0           0
 11  B        0           0
 12  B        0           0
 13  B        0           0


## Real Q-learning training

Train a tabular Q-agent on the wrapped env (seeded for reproducibility). The
agent's state is `(rule_match, open_A, open_B)`; the open bits are always 0, so
it learns purely from the rule. It should learn to open the A-chest exactly when
the rule fires, and wait otherwise.

In [5]:
from alpha_rule.policy_agents.q_learning import (
    q_learning_agent_builder, q_learning_agent_eval_mean_reward_success_steps,
)

train_env = HistoryToRuleWrapperBase(BurstChestEnv(), rule_list=[RULE_A], window=WINDOW)
q_table, policy = q_learning_agent_builder(train_env, total_timesteps=6000, seed=0)

print("learned policy (action 0=wait, 1=openA, 2=openB):")
print("  rule fires  (1,0,0) ->", policy((1.0, 0, 0)))
print("  rule quiet  (0,0,0) ->", policy((0.0, 0, 0)))

eval_env = HistoryToRuleWrapperBase(BurstChestEnv(), rule_list=[RULE_A], window=WINDOW)
mean_r, success, steps = q_learning_agent_eval_mean_reward_success_steps(
    (q_table, policy), eval_env, n_eval_episodes=10,
)
print(f"\nmean reward {mean_r:.2f}  success_rate {success:.2f}  mean_steps {steps:.0f}")
print("(28-step episode = 4 bursts, so 4 well-timed A-opens -> reward ~4)")

learned policy (action 0=wait, 1=openA, 2=openB):
  rule fires  (1,0,0) -> 1
  rule quiet  (0,0,0) -> 0

mean reward 4.00  success_rate 1.00  mean_steps 28
(28-step episode = 4 bursts, so 4 well-timed A-opens -> reward ~4)


## Scoring rules with `RuleSimulator`

`RuleSimulator` wires the four callables together and trains a fresh agent per
rule. It calls `gym.make(env_name)` internally, so here we point that at the
burst env. Scoring three rules shows the score reflects rule quality:

* `A A < A <` (precise A, fires only at the A-chest's ready step),
* `B B < B < < B` (precise B, fires only at the B-chest's ready step),
* `A` (loose, fires on every A, so the agent also opens too early).

The two precise rules let the agent time its opens and score near the `+4`
ceiling; the loose rule mistimes two of every three opens and scores lower.

In [6]:
from alpha_rule.evaluation import RuleSimulator, RuleStringNode
import alpha_rule.evaluation.rule_simulator as rs_mod

# Hand RuleSimulator the burst env instead of an OpenTheChests build.
rs_mod.gym.make = lambda env_name: BurstChestEnv()

def score(rule_str):
    sim = RuleSimulator(
        env_name="BurstChests-v0",
        agent_builder=q_learning_agent_builder,
        transformer=lambda e, rule: HistoryToRuleWrapperBase(e, rule, window=WINDOW),
        agent_eval=lambda agent, e: q_learning_agent_eval_mean_reward_success_steps(
            agent, e, n_eval_episodes=10,
        ),
        reward_scale=4.0,                       # episode reward caps near +4
        agent_builder_kwargs={"total_timesteps": 6000, "seed": 0},
        seed=0,
    )
    reward, success, steps = sim.evaluate(RuleStringNode(name=rule_str))
    return reward, success

RULE_B = build_n("B", 4)                        # 4 B's, detects the B-chest
LOOSE = "A"                                      # fires on every A, not just the 3rd
print("precise B rule (4 B's):", repr(RULE_B))
print(f"\n{'rule':24s} {'mean_reward':>12s} {'success':>9s}")
results = {}
for label, rule in [("precise A  " + RULE_A, RULE_A),
                    ("precise B  " + RULE_B, RULE_B),
                    ("loose A    " + LOOSE, LOOSE)]:
    r, s = score(rule)
    results[rule] = r
    print(f"{label:24s} {r:12.2f} {s:9.2f}")

assert results[RULE_A] >= 3.5 and results[RULE_B] >= 3.5
assert results[LOOSE] < results[RULE_A]
print("\nboth precise rules score near +4 (one per chest); the loose rule scores lower (ok)")

precise B rule (4 B's): 'B B < B < < B'

rule                      mean_reward   success


precise A  A A < A <             4.00      1.00


precise B  B B < B < < B         4.00      1.00


loose A    A                     2.00      1.00

both precise rules score near +4 (one per chest); the loose rule scores lower (ok)


## Does MCTS discover the rule?

The previous section scored rules we picked by hand. Now let the search pick
them. `run_self_play` (component 03's MCTS engine) builds a rule one production
at a time, scoring every leaf with the `RuleSimulator` (a freshly trained
Q-agent). No network priors are supplied, so the search is guided purely by the
RL reward. It should climb toward the high-reward precise detector.

In [7]:
from alpha_rule.mcts import run_self_play

def mean_reward_only(agent, env):
    # run_self_play wants one number per rule (tuple -> mean reward, or scalar
    # -inf when the rule never fires, which the search prunes as a dead rule).
    out = q_learning_agent_eval_mean_reward_success_steps(agent, env, n_eval_episodes=5)
    return out[0] if isinstance(out, tuple) else out

rs_mod.gym.make = lambda name: BurstChestEnv()
search_sim = RuleSimulator(
    env_name="BurstChests-v0",
    agent_builder=q_learning_agent_builder,
    transformer=lambda e, rule: HistoryToRuleWrapperBase(e, rule, window=WINDOW),
    agent_eval=mean_reward_only,
    agent_builder_kwargs={"total_timesteps": 400, "seed": 0},
    seed=0,
)

traj = run_self_play(
    grammar=g, simulator=search_sim, n_simulations=30, depth_limit=6,
    temperature=0.0, leaf_eval_mode="simulator", rng=np.random.default_rng(0),
)

print("MCTS builds the rule one production at a time")
print("(reward = score of that partial rule; top action = the MCTS visit favourite):\n")
print(f"{'partial rule':16s} -> {'next rule':22s} {'reward':>7s}  top action")
for s in traj.steps:
    top = max(s.visit_pi, key=s.visit_pi.get)
    print(f"{s.state:16s} -> {s.next_state:22s} {s.reward:7.2f}  {top} ({s.visit_pi[top]:.0%})")

discovered = traj.steps[-1].next_state.replace("<END>", "").strip()
print(f"\ndiscovered rule: {discovered!r}  (reward {traj.steps[-1].reward:.2f})")
print("starting from nothing, MCTS recovered a precise detector from the reward alone")

MCTS builds the rule one production at a time
(reward = score of that partial rule; top action = the MCTS visit favourite):

partial rule     -> next rule               reward  top action
<ROOT>           -> A                         1.00  A (93%)
A                -> A A                       3.00  A (91%)
A A              -> A A <                     3.00  < (86%)
A A <            -> A A < A                   4.00  A (81%)
A A < A          -> A A < A <END>             4.00  END_RULE (50%)

discovered rule: 'A A < A'  (reward 4.00)
starting from nothing, MCTS recovered a precise detector from the reward alone


## The full loop: train() learns a rule per chest

`run_self_play` searched with the reward signal alone. `train()` closes the
AlphaZero loop: each iteration runs a self-play episode (MCTS guided by the
network's priors and value), stores it in a replay buffer, and trains the
network. One `train()` run commits to a single chest, because that one rule
already captures all the reward it can see.

But there are two rules to find, one per chest. The wrapper and eval were built
for exactly this: pass the already-found rules as earlier columns of the
observation, and the eval reads the new candidate as the **last** column,
suppressing its firing whenever a context rule already fired. So re-discovering
the same chest scores nothing extra (an exact repeat scores `-inf` and is
pruned); the only way to gain reward is to detect the **other** chest. The outer
loop runs `train()`, adds the rule it finds to the context, and runs `train()`
again. (A slightly higher Q-learning `epsilon` lets the agent discover the
second chest's open action in the now-larger state.)

In [8]:
from alpha_rule.training import train, play

def reward_scalar(agent, env):
    out = q_learning_agent_eval_mean_reward_success_steps(agent, env, n_eval_episodes=6)
    return out[0] if isinstance(out, tuple) else out

def which_chest(rule):
    # Wrap the rule alone and report which chest's ready step it fires on.
    e = BurstChestEnv()
    w = HistoryToRuleWrapperBase(e, [rule], window=WINDOW)
    m, _ = w.reset()
    fires = set()
    for _ in range(14):
        if m[0]:
            fires.add(e._pos - 1)        # position just emitted
        m, _, _, _, _ = w.step(0)
    fires = {p % 7 for p in fires}
    a, b = A_READY_POS in fires, B_READY_POS in fires
    return "A-chest" if (a and not b) else ("B-chest" if (b and not a) else "mixed")

rs_mod.gym.make = lambda name: BurstChestEnv()
discovered_rules, logs = [], []
for rnd in range(2):
    sim = RuleSimulator(
        env_name="BurstChests-v0",
        agent_builder=q_learning_agent_builder,
        transformer=lambda e, rule, ctx=tuple(discovered_rules): HistoryToRuleWrapperBase(
            e, list(ctx) + [rule], window=WINDOW),         # found rules first, candidate last
        agent_eval=reward_scalar,
        reward_scale=4.0 * (len(discovered_rules) + 1),
        agent_builder_kwargs={"total_timesteps": 600, "seed": 0, "epsilon": 0.3},
        seed=0,
    )
    log = train(
        grammar=g, expensive_simulator=sim,
        n_iterations=6, n_simulations=20, depth_limit=6,
        max_len=16, d_model=16, nhead=2, num_layers=1,
        buffer_warmup=1, train_steps_per_iteration=8, seed=0,
    )
    found = (log.best_formula or "").replace("<END>", "").strip()
    print(f"round {rnd + 1}: context={discovered_rules}")
    print(f"         found {found!r}  ({which_chest(found)}, set reward {log.best_reward:.2f}, "
          f"final NN loss {log.iterations[-1].train_total:.3f})")
    discovered_rules.append(found)
    logs.append(log)

print(f"\ndiscovered rule set: {discovered_rules}")

round 1: context=[]
         found 'B'  (B-chest, set reward 3.00, final NN loss 0.742)


round 2: context=['B']
         found 'A A < A <'  (A-chest, set reward 5.00, final NN loss 0.690)

discovered rule set: ['B', 'A A < A <']


In [9]:
from alpha_rule.evaluation import NeuralEvaluator
from alpha_rule.nn import GrammarTokenizer, AllenFormulaNet

# One detector per chest?
chests = {which_chest(r) for r in discovered_rules}
assert chests == {"A-chest", "B-chest"}, f"expected one rule per chest, got {chests}"

# The NN trains in each round: the loss falls and the policy head shifts toward
# the chest that round commits to.
for r, lg in enumerate(logs):
    losses = [it.train_total for it in lg.iterations]
    print(f"round {r + 1} NN loss: {losses[0]:.3f} -> {losses[-1]:.3f}")
fresh = AllenFormulaNet(GrammarTokenizer(g), d_model=16, nhead=2, num_layers=1, max_len=16)
untrained = NeuralEvaluator(fresh, g, max_len=16).evaluate(g.root()).priors
trained = NeuralEvaluator(logs[0].model, g, max_len=16).evaluate(g.root()).priors
print("round-1 root priors  untrained:", {k: round(v, 2) for k, v in untrained.items()})
print("round-1 root priors  trained:  ", {k: round(v, 2) for k, v in trained.items()})

print("\nThe outer loop found one detector per chest: re-finding the first rule")
print("scores nothing (the eval suppresses a candidate that only fires when a")
print("context rule already did), so the second search is steered to the other")
print("chest: the As rule and the Bs rule (ok)")

round 1 NN loss: 0.911 -> 0.742
round 2 NN loss: 0.899 -> 0.690
round-1 root priors  untrained: {'A': 0.47, 'B': 0.53}
round-1 root priors  trained:   {'A': 0.41, 'B': 0.59}

The outer loop found one detector per chest: re-finding the first rule
scores nothing (the eval suppresses a candidate that only fires when a
context rule already did), so the second search is steered to the other
chest: the As rule and the Bs rule (ok)


## Basic checks

Asserts mirroring the unit tests. `ok` means the backend behaves.

In [10]:
from alpha_rule.evaluation import Evaluator

# Env readiness schedule: each chest ready once per burst (every 7 steps).
e = BurstChestEnv(); e.reset()
ready_a, ready_b = [], []
for _ in range(21):
    ready_a.append(e.get_otc().boxes[0].is_ready())
    ready_b.append(e.get_otc().boxes[1].is_ready())
    e.step(0)
assert sum(ready_a) == 3 and sum(ready_b) == 3        # 3 bursts in 21 steps
assert ready_a[A_READY_POS] and ready_b[B_READY_POS]

# RuleSimulator is an Evaluator, builds its env once, and forwards kwargs+seed.
rs_mod.gym.make = lambda name: BurstChestEnv()
seen = {}
def _spy_builder(env, **kw):
    seen.update(kw)
    return q_learning_agent_builder(env, **kw)
sim = RuleSimulator(
    "BurstChests-v0", _spy_builder,
    lambda e, rule: HistoryToRuleWrapperBase(e, rule, window=WINDOW),
    lambda agent, e: q_learning_agent_eval_mean_reward_success_steps(agent, e, n_eval_episodes=5),
    reward_scale=4.0,
    agent_builder_kwargs={"total_timesteps": 400, "seed": 1},
)
assert isinstance(sim, Evaluator)
assert sim._env is None                                # lazy: no env yet
sim.evaluate(RuleStringNode(name=RULE_A + " <END>"))   # <END> is stripped
assert sim._env is not None
assert seen == {"total_timesteps": 400, "seed": 1}     # builder kwargs forwarded
assert sim.reward_scale == 4.0

# The rule wrapper fires exactly at the A-chest ready step.
base = BurstChestEnv()
w = HistoryToRuleWrapperBase(base, rule_list=[RULE_A], window=WINDOW)
m, _ = w.reset()
fired_when_ready = []
for _ in range(14):
    fired_when_ready.append(bool(m[0]) == bool(base.get_otc().boxes[0].is_ready()))
    m, _, _, _, _ = w.step(0)
assert all(fired_when_ready)                           # match bit == A readiness, every step

print("ok")

ok


## Full unit tests

```
python -m alpha_rule.tests.run_tests test_rule_simulator
python -m alpha_rule.tests.run_tests test_wrappers
python -m alpha_rule.tests.run_tests test_incremental_matrix
python -m alpha_rule.tests.run_tests test_q_learning_builder
```

These need the `[rl]` extra (`gymnasium`); without it `run_tests` skips the
modules. They use fakes throughout, so they do not need OpenTheChests. For a
real reward signal, install OpenTheChests separately and pass its env id as
`RuleSimulator(env_name=...)`.